# Search rounds and arms

What `standard` and `deep` are *designed* to do beyond round 1:

| depth | rounds | arms from round 2 |
|---|---|---|
| rapid | 1 | none |
| standard | 2 | reformulate, diversity |
| deep | 3 | reformulate, diversity, snowball (citations forward + backward), suggest |

This notebook drives that loop for real and reports, per round: documents acquired,
documents screened, new confidently-relevant found, the cost per marginal
confidently-relevant document, which arms fired, and why the loop stopped.

> ### Read this before running
>
> **1. The orchestrator does not currently run this loop.** `run_deep_rounds` and
> `should_escalate` are defined and unit-tested in `search_loop.py` but have **no caller
> in `src/`**. A composed chain (`orchestration_plan.compose`) contains exactly one
> `acquire` step and one `screen_abstract` step, and `harness._run_acquire` calls
> `run_search` once. So in a normal run today, every depth does **one** round; the extra
> rounds only happen if acquire is invoked again on the same scope (a steering-driven
> re-run or additive segment re-entry), because `run_search` derives `round_index` from
> the number of existing coverage rows.
>
> This notebook therefore shows the loop **as designed**, by calling `run_deep_rounds`
> directly — which is also the harness you would want when wiring it up.
>
> **2. This costs real money.** Unlike `search_caps_by_depth.ipynb`, the arms need
> graded exemplars, so screening has to run between rounds. Screening bills
> `documents x SCREEN_REPS (3) x (1 + retry)`. At the current caps a deep round can
> acquire up to 400 documents, so a 3-round deep run is on the order of 1,200 documents
> and ~3,600 model calls. Set `DEPTH` and read the estimate the notebook prints before
> you commit to it.

In [1]:
import os
import sys
import time
import uuid
from datetime import UTC, datetime
from pathlib import Path

from dotenv import load_dotenv

REPO = Path.cwd()
while not (REPO / "backend").is_dir() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO / "backend" / "src"))
load_dotenv(REPO / ".env")

MISSING = [k for k in ("OPENALEX_API_KEY", "OVERTON_API_KEY", "OPENAI_API_KEY")
           if not os.environ.get(k)]
print("repo:", REPO)
print("database:", (os.environ.get("DATABASE_URL") or "UNSET").rsplit("/", 1)[-1])
print("missing keys:", MISSING or "none")

repo: /Users/rosie.oxbury/Documents/git_repos/policy_atlas
database: policy_atlas
missing keys: none


In [4]:
from sqlalchemy import select

from policy_atlas.core import events
from policy_atlas.core.db import get_engine
from policy_atlas.core.schema import (
    evidence_scope,
    project,
    project_source_snapshot,
    runs,
    search_coverage_record,
)
from policy_atlas.evidence_base.assess.screen import ScreenContext, screen_sources
from policy_atlas.evidence_base.assess.screening_backend import OpenAIScreeningBackend
from policy_atlas.evidence_base.sourcing import search_generation, search_live
from policy_atlas.evidence_base.sourcing.acquire import AcquireContext
from policy_atlas.evidence_base.sourcing.search_loop import (
    CONFIDENT_FLOOR,
    DEPTH_CONSTANTS,
    SHORT_CIRCUIT_RATE,
    #TARGET_CONFIDENT_RELEVANT, # RO: This was set to 100, which makes it unlikely that the search will progress beyond 1 round
    confident_relevant_count,
    run_deep_rounds,
    run_search,
)

QUERY = (
    "interventions to reduce consumption of high fat, sugar, and salt (HFSS) foods"
)
DEPTH = "deep"            # "standard" for the 2-round, reformulate-only version

# The stop target, passed EXPLICITLY to run_deep_rounds below. Rebinding the
# imported TARGET_CONFIDENT_RELEVANT name would NOT work: `target` is a default
# argument, so Python froze it to the module value (100) at import time. Set it
# very high to disable the target stop and let the loop run to round_cap.
TARGET_CONFIDENT_RELEVANT = 1000
TARGET = 1000

constants = DEPTH_CONSTANTS[DEPTH]
per_round = (constants["record_cap_per_backend"] or 0) * 2
print(f"depth={DEPTH}  round_cap={constants['round_cap']}  arms={sorted(constants['arms'])}")
print(f"stop target: {TARGET} confident-relevant (confidence >= {CONFIDENT_FLOOR})"
      + ("" if TARGET == TARGET_CONFIDENT_RELEVANT
         else f"   [overriding the as-built {TARGET_CONFIDENT_RELEVANT}]"))
print(f"short-circuit if a round yields < 1 new confident-relevant per "
      f"{int(1 / SHORT_CIRCUIT_RATE)} screened")
print(f"\nworst case: ~{per_round} docs/round x {constants['round_cap']} rounds "
      f"= ~{per_round * constants['round_cap']} docs screened "
      f"(~{per_round * constants['round_cap'] * 3} model calls)")

depth=deep  round_cap=3  arms=['diversity', 'reformulate', 'snowball', 'suggest']
stop target: 1000 confident-relevant (confidence >= 0.7)
short-circuit if a round yields < 1 new confident-relevant per 50 screened

worst case: ~400 docs/round x 3 rounds = ~1200 docs screened (~3600 model calls)


## Set up one project and scope

Every round writes into the same project and scope — that is what makes them *rounds*
rather than unrelated runs. `run_search` reads `round_index` from the number of coverage
rows already present for the scope, so round 2 knows it is round 2 because round 1 left
a row behind.

In [5]:
engine = get_engine()

now = datetime.now(UTC)
PROJECT_ID, SCOPE_ID = uuid.uuid4(), uuid.uuid4()
with engine.begin() as conn:
    conn.execute(project.insert().values(
        project_id=PROJECT_ID, name=f"rounds-{DEPTH}-{now:%Y%m%d-%H%M%S}",
        status="active", created_at=now, updated_at=now,
    ))
    conn.execute(evidence_scope.insert().values(
        evidence_scope_id=SCOPE_ID, project_id=PROJECT_ID, intent=QUERY,
        context={"search": {"depth": DEPTH}}, created_at=now,
    ))

ACQUIRE_CONTEXT = AcquireContext(
    scope_id=SCOPE_ID, intent=QUERY, context={"search": {"depth": DEPTH}}
)
SCREEN_CONTEXT = ScreenContext(scope_id=SCOPE_ID, intent=QUERY, context={})
generation = search_generation.OpenAISearchGenerationBackend()
screening = OpenAIScreeningBackend()

ROUND_RUNS = []     # one run_id per round, so events can be attributed per round
print("project:", PROJECT_ID)
print("scope:  ", SCOPE_ID)

project: 7e2a25f2-0ef7-4ee5-9ca0-de33541ebd20
scope:   10143fcc-59b0-44f3-84a9-da33711fc2f8


In [6]:
def new_run(conn):
    run_id = uuid.uuid4()
    conn.execute(runs.insert().values(
        run_id=run_id, project_id=PROJECT_ID, status="running",
        started_at=datetime.now(UTC),
    ))
    return run_id


def acquire_round():
    """One acquire round. run_search picks its own round_index from coverage rows."""
    with engine.begin() as conn:
        run_id = new_run(conn)
        ROUND_RUNS.append(run_id)
        counts = run_search(
            conn,
            project_id=PROJECT_ID,
            run_id=run_id,
            context=ACQUIRE_CONTEXT,
            backends=search_live.live_search_backends(),
            generation_backend=generation,
        )
    print(f"  acquire  round={counts['search']['round_index']}  "
          f"acquired={counts['acquired']}  returned={counts['results_returned']}  "
          f"over_cap={counts['dropped_over_cap']}")
    return counts


def screen_round():
    """One stage-1 screen round over whatever is unscreened in the scope."""
    with engine.begin() as conn:
        counts = screen_sources(
            conn,
            project_id=PROJECT_ID,
            run_id=ROUND_RUNS[-1],
            context=SCREEN_CONTEXT,
            screening_backend=screening,
        )
    print(f"  screen   screened={counts.get('screened')}  "
          f"relevant={counts.get('relevant')}")
    return counts

## Round 1

The first round is the plain fan-out — no arms, because `round_index` is 1. It has to be
screened before round 2 can reformulate, since the reformulate prompt is built from
*graded exemplars*: documents this run screened as relevant, and ones it screened out.

In [7]:
started = time.monotonic()
print("round 1")
r1_acquire = acquire_round()
r1_screen = screen_round()

with engine.begin() as conn:
    confident_after_r1 = confident_relevant_count(
        conn, project_id=PROJECT_ID, scope_id=SCOPE_ID
    )
print(f"\nconfident-relevant after round 1: {confident_after_r1} "
      f"(target {TARGET})")
print(f"elapsed: {time.monotonic() - started:.0f}s")

round 1
2026-08-05 17:21:13 [info     ] search_generation.queries.usage cached_tokens=0 completion_tokens=125 prompt_tokens=634 total_tokens=759
2026-08-05 17:21:48 [warning  ] search.http_retry              attempt=1 host=api.openalex.org status_code=None
2026-08-05 17:22:18 [info     ] acquire.capped                 backend=openalex cap=200 dropped=308 kept=200 project_id=7e2a25f2-0ef7-4ee5-9ca0-de33541ebd20 run_id=b7b55ff6-e3fd-4105-9e15-e7a927171d85
2026-08-05 17:22:18 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=inraefr-ec7857d2cdb562679224c0fdfd17c19b cap=50 tag_count=72
2026-08-05 17:22:18 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=stateofnebraska-e9fb3ab18bfc3d7afec34cb8dfc292e9 cap=50 tag_count=54
2026-08-05 17:22:18 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=ukparliament_select-9bb91dbc7fc53b9832005d05988af706 cap=50 tag_count=55
2026-08-05 17:22:18 [warning  ] acquire.tags_tru

## Rounds 2+

`run_deep_rounds` drives the rest: acquire, screen, evaluate, repeat. It stops on the
first of

- `target_reached` — enough confidently-relevant documents
- `short_circuit` — the round's yield fell below 1 per 50 screened, so searching harder
  is not paying
- `budget_exhausted` — round cap reached

Two things that will bite if you change constants to experiment:

**`target` is a default argument, so the module value is frozen at import.**
`run_deep_rounds(..., target: int = TARGET_CONFIDENT_RELEVANT)` captures the constant
*once*, when `search_loop` is first imported. Rebinding `TARGET_CONFIDENT_RELEVANT` in a
notebook cell — or even assigning `search_loop.TARGET_CONFIDENT_RELEVANT = 1000` — has no
effect on it. That is why the cell below passes `target=TARGET` explicitly. To disable the
target stop entirely and force the loop to `round_cap`, set `TARGET` to something
unreachable (e.g. `10_000`) in the config cell and re-run from there.

**`round_cap` is always deep's.** `run_deep_rounds` reads
`DEPTH_CONSTANTS["deep"]["round_cap"]` directly, so it allows 3 rounds even when
`DEPTH = "standard"` — standard's own `round_cap` of 2 is not consulted.

In [8]:
rounds_run = 1


def acquire_round_bounded():
    global rounds_run
    rounds_run += 1
    print(f"round {rounds_run}")
    return acquire_round()


with engine.connect() as loop_conn:
    summary = run_deep_rounds(
        loop_conn,
        project_id=PROJECT_ID,
        scope_id=SCOPE_ID,
        acquire_round=acquire_round_bounded,
        screen_round=screen_round,
        start_round=2,
        target=TARGET,
    )
    loop_conn.commit()

print(f"\nstop_condition       = {summary['stop_condition']}")
print(f"confident_relevant   = {summary['confident_relevant']}")
print(f"rounds in loop       = {len(summary['rounds'])}")
print(f"wall clock           = {summary['wall_clock_s']:.0f}s")

round 2
2026-08-05 17:24:50 [info     ] search_generation.reformulate.usage cached_tokens=0 completion_tokens=120 prompt_tokens=2713 total_tokens=2833
2026-08-05 17:25:07 [info     ] search_generation.suggest.usage cached_tokens=0 completion_tokens=299 prompt_tokens=1930 total_tokens=2229
2026-08-05 17:25:11 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=asauk-7859e2aad73310e47c048f6f8b61cd3c cap=50 tag_count=55
2026-08-05 17:25:11 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=paho-71324655d322a82d6643ce8d88a4df8a cap=50 tag_count=61
2026-08-05 17:25:11 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=paho-67fc0762396e54ff0ed3aad9d8bfcb1d cap=50 tag_count=54
2026-08-05 17:25:11 [warning  ] acquire.tags_truncated         backend=overton backend_record_id=govsg-1e8f3b82e9893b8ab6cc182658926d4a cap=50 tag_count=53
2026-08-05 17:25:13 [info     ] embed.summary                  already_embedded=400 budg

## Per-round yield

`cost_per_marginal_confident_relevant` is documents screened divided by *new*
confidently-relevant found. It should climb every round — the cheap finds come first.
When it climbs past what a confident-relevant document is worth to you, that is the
round the loop should have stopped at.

In [9]:
rows = [{
    "round": 1,
    "docs_screened": r1_screen.get("screened"),
    "new_confident_relevant": confident_after_r1,
    "cost_per_marginal_confident_relevant": (
        r1_screen.get("screened") / confident_after_r1 if confident_after_r1 else None
    ),
}] + list(summary["rounds"])

print("{:>6} {:>14} {:>12} {:>18}".format(
    "round", "docs screened", "new c-rel", "cost per new c-rel"))
for row in rows:
    cost = row["cost_per_marginal_confident_relevant"]
    print("{:>6} {:>14} {:>12} {:>18}".format(
        row["round"],
        row["docs_screened"],
        row["new_confident_relevant"],
        "-" if cost is None else f"{cost:.1f}",
    ))

total_screened = sum(r["docs_screened"] or 0 for r in rows)
print(f"\ntotal screened across rounds: {total_screened}"
      f"  (~{total_screened * 3} model calls)")

 round  docs screened    new c-rel cost per new c-rel
     1            400          309                1.3
     2            203          173                1.2
     3             34           31                1.1

total screened across rounds: 637  (~1911 model calls)


## Which arms actually fired

Every executed call emits a `search.executed` event carrying its `verb` and
`query_origin`. Those two fields identify the arm:

| arm | signature |
|---|---|
| round-1 fan-out | `verb=search`, origin `generated` / `variant_sr` / `variant_rct` |
| reformulate | `verb=search`, origin `generated` at round >= 2 |
| diversity | `verb=search`, origin `verbatim`, query == the intent |
| snowball forward | `verb=fetch_citations` |
| snowball backward | `verb=fetch_references` |
| suggest | `verb=lookup_dois` / `lookup_title` |

Calls are attributed to a round by the run id each round used.

In [10]:
with engine.connect() as conn:
    log = events.read(conn, PROJECT_ID)

run_to_round = {str(run_id): index + 1 for index, run_id in enumerate(ROUND_RUNS)}
by_round = {}
for event in log:
    if event["event_type"] != "search.executed":
        continue
    round_index = run_to_round.get(str(event.get("run_id")), "?")
    key = (round_index, event["payload"]["verb"], event["payload"]["query_origin"])
    entry = by_round.setdefault(key, {"calls": 0, "records": 0})
    entry["calls"] += 1
    entry["records"] += event["payload"].get("result_count") or 0

print("{:>6} {:<18} {:<16} {:>7} {:>9}".format(
    "round", "verb", "query_origin", "calls", "records"))
for (round_index, verb, origin), entry in sorted(by_round.items(), key=lambda kv: str(kv[0])):
    print("{:>6} {:<18} {:<16} {:>7} {:>9}".format(
        round_index, verb, origin, entry["calls"], entry["records"]))

arms_seen = {verb for (_r, verb, _o) in by_round}
print("\nverbs seen:", sorted(arms_seen))
for verb, label in (("fetch_citations", "snowball forward"),
                    ("fetch_references", "snowball backward"),
                    ("lookup_dois", "suggest (DOI grounding)"),
                    ("lookup_title", "suggest (title grounding)")):
    print(f"  {label:28s} {'yes' if verb in arms_seen else 'NO'}")

 round verb               query_origin       calls   records
     1 search             generated              5       315
     1 search             paraphrase             2       200
     1 search             variant_rct            5        98
     1 search             variant_sr             5       129
     1 search             verbatim               1       100
     2 fetch_citations    snowball_forward       5         8
     2 fetch_references   snowball_backward       1        32
     2 lookup_title       suggestion_title       6         1
     2 search             generated              4       103
     2 search             paraphrase             2       200
     2 search             verbatim               1        15
     3 fetch_citations    snowball_forward       5         8
     3 fetch_references   snowball_backward       1        32
     3 lookup_title       suggestion_title       6         9
     3 search             generated              4        32
     3 search         

## Corpus at the end

In [11]:
with engine.connect() as conn:
    total = conn.execute(
        select(project_source_snapshot)
        .where(project_source_snapshot.c.project_id == PROJECT_ID)
    ).fetchall()
    coverage = conn.execute(
        select(search_coverage_record)
        .where(search_coverage_record.c.project_id == PROJECT_ID)
        .order_by(search_coverage_record.c.created_at)
    ).fetchall()

print(f"documents in corpus: {len(total)}")
print(f"coverage rows (one per round): {len(coverage)}")
for index, row in enumerate(coverage, start=1):
    print(f"  round {index}: stop={row.stop_condition}  adequacy={row.adequacy_verdict}")
print(f"\nproject_id = {PROJECT_ID}")

documents in corpus: 637
coverage rows (one per round): 3
  round 1: stop=completed  adequacy=adequate
  round 2: stop=completed  adequacy=adequate
  round 3: stop=re_searched_still_thin  adequacy=adequate

project_id = 7e2a25f2-0ef7-4ee5-9ca0-de33541ebd20


## What to take from this

- **Did the arms fire?** If `fetch_citations` / `lookup_dois` never appear, the deep
  round had nothing to work with — snowball seeds from *confidently-relevant OpenAlex*
  records, and suggest needs the model to propose papers that ground. Zero here is a
  finding about the screening yield, not about the arms.
- **Does the yield curve justify the rounds?** If round 3's
  `cost_per_marginal_confident_relevant` is many times round 1's, the extra round is
  buying very little for its screening bill.
- **Which stop fired?** `budget_exhausted` means the loop ran out of rounds with the
  target unmet; `short_circuit` means it gave up because yield collapsed;
  `target_reached` means it found enough. With `TARGET_CONFIDENT_RELEVANT` at 100 you
  should expect `budget_exhausted` most of the time — that is deliberate, so the
  ground-truth eval can measure recall without the target truncating the search.
- **Remember this loop is not wired into the orchestrator.** Anything you conclude here
  is about the design, not about what a user's run does today.